In [ ]:
import torch

from mpc import mpc
from mpc.mpc import QuadCost, LinDx, GradMethods
from mpc.env_dx import cartpole

import numpy as np
import numpy.random as npr
import torch.nn as nn
import torch
import torch.nn.functional as F

import matplotlib.pyplot as plt

import os
import io
import base64
import tempfile
from IPython.display import HTML

from tqdm import tqdm

%matplotlib inline

# Utils

In [ ]:
device = "cpu"
print(device)

In [ ]:
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

set_seed(42)

In [ ]:
# 用于生成均匀分布
def uniform(shape, low, high):
    r = high-low
    return torch.rand(shape)*r+low

def init_states(n_batch, angle=180):
    th = uniform(n_batch, -2*np.pi, 2*np.pi)
    thdot = uniform(n_batch, -.5, .5)
    x = uniform(n_batch, -0.5, 0.5)
    xdot = uniform(n_batch, -0.5, 0.5)
    xinit = torch.stack((x, xdot, torch.cos(th), torch.sin(th), thdot), dim=1)
    return xinit

In [ ]:
def make_video(idx):
    vid_fname = f'cartpole{idx}.mp4'

    if os.path.exists(vid_fname):
        os.remove(vid_fname)
        
    t_dir = 'D:/Docs/code_lib/graduation_test/cartpole_pic'

    cmd = 'ffmpeg -r 16 -f image2 -i {}/frame_%03d.png -vcodec libx264 -crf 25 -vf "pad=ceil(iw/2)*2:ceil(ih/2)*2" -pix_fmt yuv420p {}'.format(
        t_dir, vid_fname
    )
    print(cmd)
    os.system(cmd)
    print('Saving video to: {}'.format(vid_fname))

# Init

In [ ]:
dx = cartpole.CartpoleDx()


# q, p = dx.get_true_obj()
# print(q)
# print(p)
# q_ori = torch.tensor([0.1, 0.1, 1., 1., 0.1])
# p_ori = torch.tensor([0., 0., -1., 0., 0.])
# rand_q_bias = torch.randn_like(q_ori) * 0.01
# rand_p_bias = torch.randn_like(p_ori) * 0.01
# q = q_ori + rand_q_bias
# p_ = p_ori + rand_p_bias
# q = nn.Parameter(q)
# p_ = nn.Parameter(p_)
# control_penalty = 0.001
# qxu = torch.cat(q, control_penalty*torch.ones(dx.n_ctrl))
# pxu = torch.cat(p_, torch.zeros(dx.n_ctrl))

# # 每个batch和每个timestamp的运行损失都是一样的，这里就是在构造这样的Q和p
# Q = torch.diag(qxu).unsqueeze(0).unsqueeze(0).repeat(
#     mpc_T, n_batch, 1, 1
# )
# p = pxu.unsqueeze(0).repeat(mpc_T, n_batch, 1)

t_dir = "D:/Docs/code_lib/graduation_test/cartpole_pic"

In [ ]:
def run_mpc(Q, p, x, u_init, save_fig=False):
    x_list = []
    u_list = []
    V_list = []
    for t in tqdm(range(T)):
        nominal_states, nominal_actions, nominal_objs = mpc.MPC(
            dx.n_state, dx.n_ctrl, mpc_T,
            u_init=u_init,                                  # u_init的作用: warm-start，表示对控制序列的初始猜测 (用上一个时刻的预测结果)，可以加速收敛
            # u_lower=dx.lower, u_upper=dx.upper,
            lqr_iter=50,
            verbose=0,
            exit_unconverged=False,
            detach_unconverged=False,
            linesearch_decay=dx.linesearch_decay,
            max_linesearch_iter=dx.max_linesearch_iter,
            grad_method=GradMethods.AUTO_DIFF,
            eps=1e-2,
        )(x, QuadCost(Q, p), dx)
        
        next_action = nominal_actions[0]
        u_init = torch.cat((nominal_actions[1:], torch.zeros(1, n_batch, dx.n_ctrl)), dim=0) # u_init shape (mpc_T, batch_size, 1)
        u_init[-2] = u_init[-3] # 不知道为什么这样写，猜想应该是u[-1] = u[-2]才对

        x_list.append(nominal_states) # x, dim=3, shape=(T, batch_size, n_state)
        u_list.append(nominal_actions) # u, dim=3, shape=(T, batch_size, n_ctrl)
        V_list.append(nominal_objs) # V, dim=1, shape=(batch_size, )

        x = dx(x, next_action) # 往前走一步

        # 以下都是用来可视化的
        if save_fig:
            n_col = 4
            n_row = n_batch // n_col
            fig, axs = plt.subplots(n_row, n_col, figsize=(3*n_col,3*n_row), gridspec_kw = {'wspace':0, 'hspace':0})
            axs = axs.reshape(-1)
            for i in range(n_batch):
                dx.get_frame(x[i], ax=axs[i])
                axs[i].get_xaxis().set_visible(False)
                axs[i].get_yaxis().set_visible(False)
            fig.tight_layout()
            fig.savefig(os.path.join(t_dir, 'frame_{:03d}.png'.format(t)))
            plt.close(fig)
            
    return x_list, u_list, V_list

# RDP

In [ ]:
def cost_cartpole(x, u, Q, p, R, is_terminal):
    if not is_terminal:
        return 0.5 * torch.matmul(torch.matmul(x.unsqueeze(1), Q), x.unsqueeze(2)).squeeze(1).squeeze(1) + \
               0.5 * torch.matmul(torch.matmul(u.unsqueeze(1), R), u.unsqueeze(2)).squeeze(1).squeeze(1) + \
               torch.matmul(x.unsqueeze(1), p).squeeze(1)
    else:
        return 0.5 * torch.matmul(torch.matmul(x.unsqueeze(1), Q), x.unsqueeze(2)).squeeze(1).squeeze(1) + \
               torch.matmul(x.unsqueeze(1), p).squeeze(1)

def cal_V(states, actions, q, p, R):
    Q = torch.diag(q)
    V_list = []

    for t in range(mpc_T):
        V_now = 0
        for i in range(T):
            if i == T - 1:
                V_now += cost_cartpole(states[t][i], actions[t][i], Q, p, R, True)
            else:
                V_now += cost_cartpole(states[t][i], actions[t][i], Q, p, R, False)
        V_list.append(V_now)
    return V_list

def RDP_criteria_cartpole(VN_list, x_list, u_list, q, p, R, alpha, mpc_T, func, test=False, log_path=None):
    """
    Input:
        VN_list: list of the cost function value
        x_list: list of the state
        u_list: list of the control
        alpha: the weight of the running cost term
        func: the function used to incoporate the RDP inequality into the loss function
        test: whether to print the RDP value
    Return:
        RDP criteria
    """
    Q = torch.diag(q)
    loss = 0
    for i in range(mpc_T - 1):
        RDP = VN_list[i+1] + alpha * cost_cartpole(x_list[i], u_list[i], Q, p, R, False) - VN_list[i] # Wish RDP <= 0

        if test:
            if log_path is not None:
                with open(log_path, 'a') as f:
                    f.write(f'RDP{i}: {RDP}\n')
        loss += func(RDP) 
        
    return loss

# Train

In [ ]:
epochs = 50
n_batch, T, mpc_T = 16, 100, 100
q_ori = torch.tensor([0.1, 0.1, 1., 1., 0.1])
p_ori = torch.tensor([0., 0., -1., 0., 0.])
rand_q_bias = torch.randn_like(q_ori) * 0.5
rand_p_bias = torch.randn_like(p_ori) * 0.5
q = q_ori + rand_q_bias
p_ = p_ori + rand_p_bias
q = nn.Parameter(q)
p_ = nn.Parameter(p_)
control_penalty = 0.001
R = torch.ones(1) * control_penalty
R = R.unsqueeze(0)

n_state = 5
n_ctrl = 1

log_path = './log.txt'
optimizer = torch.optim.Adam([q, p_], lr=0.01)
for epoch in range(epochs):
    with open(log_path, 'a') as f:
        f.write(f'Epoch: {epoch}\n')
        f.write(f'q: {q}\n')
        f.write(f'p: {p_}\n')
    # construct the loss function
    qxu = torch.cat((q.detach(), control_penalty*torch.ones(n_ctrl)))
    pxu = torch.cat((p_.detach(), torch.zeros(n_ctrl)))
    Q = torch.diag(qxu).unsqueeze(0).unsqueeze(0).repeat(mpc_T, n_batch, 1, 1)
    p = pxu.unsqueeze(0).repeat(mpc_T, n_batch, 1)
    
    x_init = init_states(n_batch)
    u_init = None

    states, actions, objs_ori = run_mpc(Q, p, x_init, u_init)
    V_list = cal_V(states, actions, q, p_, R)
    x_list = [nom_states[0] for nom_states in states]
    u_list = [nom_actions[0] for nom_actions in actions]
    optimizer.zero_grad()
    loss = RDP_criteria_cartpole(V_list, x_list, u_list, q, p_, R, 1, mpc_T, lambda x:torch.relu(x), test=True, log_path=log_path).mean()
    with open(log_path, 'a') as f:
        f.write(f'Loss: {loss}\n')
    loss.backward()
    optimizer.step()

# 画action用

In [ ]:
# Plot actions
for t in tqdm(range(T)):
    fig, axs = plt.subplots(n_row, n_col, figsize=(3*n_col,3*n_row), gridspec_kw = {'wspace':0, 'hspace':0})
    axs = axs.reshape(-1)
    for i in range(n_batch):
        axs[i].plot(action_history[:,i], color='k')
        axs[i].set_ylim(-15, 15)
        axs[i].axvline(t, color='k', ls='--', linewidth=4)
        axs[i].get_xaxis().set_visible(False)
        axs[i].get_yaxis().set_visible(False)
    fig.tight_layout()
    fig.savefig(os.path.join(t_dir, 'actions_{:03d}.png'.format(t)))
    plt.close(fig)
    
    f1 = os.path.join(t_dir, 'frame_{:03d}.png'.format(t))
    f2 = os.path.join(t_dir, 'actions_{:03d}.png'.format(t))
    f_out = os.path.join(t_dir, '{:03d}.png'.format(t))
    os.system(f'convert {f1} {f2} +append -resize 1200x {f_out}')

# 画图用

In [ ]:
vid_fname = 'cartpole.mp4'

if os.path.exists(vid_fname):
    os.remove(vid_fname)
    
t_dir = 'D:/Docs/code_lib/graduation_test/cartpole_pic'

cmd = 'ffmpeg -r 16 -f image2 -i {}/frame_%03d.png -vcodec libx264 -crf 25 -vf "pad=ceil(iw/2)*2:ceil(ih/2)*2" -pix_fmt yuv420p {}'.format(
    t_dir, vid_fname
)
print(cmd)
os.system(cmd)
print('Saving video to: {}'.format(vid_fname))